# Data Cleaning Pipeline

This notebook documents the cleaning steps applied to the raw Boston 311 datasets (2015 and 2025) to produce the analysis-ready CSVs used throughout the project.

**Input:** `data/311_2015.csv`, `data/311_2025.csv`  
**Output:** `data/cleaned2015.csv`, `data/cleaned2025.csv`

> The cleaned CSVs are already generated. To regenerate them, uncomment the final cell.

In [ ]:
import pandas as pd
from api311 import Year

## Cleaning Function

The following filters are applied in sequence. Each step removes requests that are either:
- **Not actionable complaints** (e.g. still open, bulk pickups, tree maintenance)
- **Outside our analysis scope** (e.g. parking enforcement, recycling, graffiti, animal control, signs & signals, Mayor's Hotline)

| Step | Filter | Reason |
|------|--------|---------|
| 1 | `case_status == 'Closed'` | Only include resolved requests |
| 2 | Exclude tree maintenance and bulk pickup titles | Scheduled services, not reactive complaints |
| 3 | Exclude `subject == "Mayor's 24 Hour Hotline"` | Catch-all category, not geographically meaningful |
| 4 | Exclude `type == 'Parking Enforcement'` | Enforcement action, not a service complaint |
| 5 | Exclude `reason == 'Recycling'` | Handled by separate city program |
| 6 | Exclude `type == 'Requests for Street Cleaning'` | Scheduled service, not reactive |
| 7 | Exclude `type == 'Graffiti Removal'` | Distinct program, skews neighborhood profiles |
| 8 | Exclude case titles containing `'Animal'` | Animal control is a separate department |
| 9 | Exclude `reason == 'Enforcement & Abandoned Vehicles'` | Enforcement, not service request |
| 10 | Exclude `reason == 'Signs & Signals'` | Infrastructure maintenance, not resident complaints |

In [ ]:
def clean_data(year_obj: Year) -> pd.DataFrame:
    """
    Apply sequential filters to a Year object's raw 311 data.
    Returns a cleaned DataFrame ready for analysis.
    """
    df = year_obj.data

    df = df[df["case_status"] == "Closed"]
    df = df[~df["case_title"].isin([
        "Tree Maintenance Requests",
        "Schedule a Bulk Item Pickup",
        "Schedule Bulk Item Pickup"
    ])]
    df = df[df["subject"] != "Mayor's 24 Hour Hotline"]
    df = df[df["type"] != "Parking Enforcement"]
    df = df[df["reason"] != "Recycling"]
    df = df[df["type"] != "Requests for Street Cleaning"]
    df = df[df["type"] != "Graffiti Removal"]
    df = df[~df["case_title"].str.contains("Animal", na=False, case=False)]
    df = df[df["reason"] != "Enforcement & Abandoned Vehicles"]
    df = df[df["reason"] != "Signs & Signals"]

    return df

## Cleaning Results

Record counts after cleaning (from last run):

| Dataset | Raw | After Cleaning | Removed |
|---------|-----|----------------|---------|
| 2015 | 210,083 | 121,063 | 89,020 (42%) |
| 2025 | 267,187 | 108,004 | 159,183 (60%) |

In [ ]:
# Uncomment to regenerate the cleaned CSVs from raw data

# d2015 = Year("data/311_2015.csv")
# d2025 = Year("data/311_2025.csv")

# cleaned2015 = clean_data(d2015)
# cleaned2025 = clean_data(d2025)

# print(f"2015: {len(d2015.data):,} → {len(cleaned2015):,} records")
# print(f"2025: {len(d2025.data):,} → {len(cleaned2025):,} records")

# cleaned2015.to_csv("data/cleaned2015.csv", index=False)
# cleaned2025.to_csv("data/cleaned2025.csv", index=False)
# print("Saved cleaned CSVs to data/")